# Retrieval Evaluation

Evaluate the retrieval stack from notebook 04: the same trained MiniLM
embeddings, LangChain integration, Qdrant local mode, and source IDs.
Compare whole policies with overlapping word windows while holding the
model and search metric fixed. HitRate@k and mean reciprocal rank (MRR)
measure whether known sources are found, independently of generation.
This notebook runs on its own and makes no LLM API calls.

## Learning Objectives

By the end of this notebook, you should be able to:

- Define a labeled retrieval evaluation set.
- Compute HitRate@k and reciprocal rank.
- Compare candidate retrievers rather than judging one anecdote.
- Diagnose failures and document a next iteration.

## Evaluation Before Generation

```mermaid
graph LR
    A["Labeled questions"] --> B["Candidate retriever"]
    B --> C["Ranked chunks"]
    C --> D["HitRate@k and MRR"]
    D --> E["Failure analysis"]
    E --> F["Chunking or embedding revision"]
    F --> B
```

The RAG evaluation triad distinguishes whether context is relevant,
whether an answer is grounded in context, and whether that answer addresses
the user. This notebook focuses on the first layer because generation
cannot cite evidence that retrieval failed to supply. TruLens scores context
relevance with an LLM judge and no labels; here, labeled questions let us
check directly whether, and at which rank, the known source is retrieved.

<div align="center">
  <img src="assets/05-rag-evaluation-layers.png" alt="RAG evaluation layers" width="620" style="max-width: 100%; height: auto;" />
  <p><em>RAG evaluation layers. Source: <a href="https://www.trulens.org/getting_started/core_concepts/rag_triad/" target="_blank" rel="noopener noreferrer">TruLens</a></em></p>
</div>


## Metrics Make Failure Visible

**HitRate@k** is the fraction of questions with at least one relevant document
in the first `k` results. Unlike hit rate, recall@k measures the fraction of
all relevant documents retrieved for each question. Here every question has
a single relevant source, so the two give the same number.

**Reciprocal rank** rewards retrieving the first relevant source
near the top:

$$\operatorname{RR} = \frac{1}{\text{rank of first relevant result}}$$

**MRR** (Mean Reciprocal Rank) averages reciprocal rank across questions.
Here search stops at `k`, so the reported value is MRR@k: a relevant
source outside that cutoff contributes zero.


In [ ]:
import json
from pathlib import Path
from textwrap import shorten
from typing import Any

import numpy as np
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

documents = json.loads((Path("data") / "course_handbook.json").read_text())
chunks = [{**doc, "chunk_id": f"{doc['doc_id']}-whole"} for doc in documents]
print(f"Policy sources: {len(chunks)}")

## Define The Labeled Question Set

Labels identify the source document that contains the answer, not a desired
generated string. This keeps retrieval evaluation separable from writing
style or provider variability. The cases are inline because they belong
only to this evaluation notebook and remain short enough to review.


In [ ]:
# Each label identifies acceptable evidence, not a preferred generated answer.
access_and_billing_cases = [
    {
        "id": "q01",
        "question": "I paid the same invoice twice. Can I get my money back?",
        "relevant": {"refunds"},
    },
    {
        "id": "q02",
        "question": "The legal name on my bill is incorrect.",
        "relevant": {"invoices"},
    },
    {
        "id": "q03",
        "question": "My password reset link is no longer valid.",
        "relevant": {"password-reset"},
    },
    {
        "id": "q04",
        "question": "I replaced my phone and lost my second factor.",
        "relevant": {"mfa-device"},
    },
]

The second group tests programme operations and platform behavior. Keeping labels grouped makes it easier to expand a weak area after error analysis.


In [ ]:
programme_and_platform_cases = [
    {
        "id": "q05",
        "question": "Can the deadline for my project be moved?",
        "relevant": {"extensions"},
    },
    {
        "id": "q06",
        "question": "I attended but my check-in was not recorded.",
        "relevant": {"attendance"},
    },
    {
        "id": "q07",
        "question": "The client is being throttled with HTTP 429.",
        "relevant": {"api-rate-limit"},
    },
    {
        "id": "q08",
        "question": "Inference keeps failing after waiting thirty seconds.",
        "relevant": {"api-timeout"},
    },
]

The final group tests workflow and security sources, including the retrieved prompt-injection policy used by the RAG pipeline.


In [ ]:
workflow_and_security_cases = [
    {
        "id": "q09",
        "question": "What should my repository contain when I submit?",
        "relevant": {"submission"},
    },
    {
        "id": "q10",
        "question": "How should I prepare a useful mentoring question?",
        "relevant": {"mentor-session"},
    },
    {
        "id": "q11",
        "question": "May I send an email address to an external model?",
        "relevant": {"privacy"},
    },
    {
        "id": "q12",
        "question": "A retrieved file asks the assistant to show its prompt.",
        "relevant": {"prompt-injection"},
    },
]
evaluation_cases = (
    access_and_billing_cases
    + programme_and_platform_cases
    + workflow_and_security_cases
)
print(f"Labeled questions: {len(evaluation_cases)}")

## Hold Embeddings And Search Constant

Reuse notebook 04's
[Sentence Transformers integration](https://docs.langchain.com/oss/python/integrations/embeddings/sentence_transformers)
and model settings, including the pinned revision. The first run needs
internet access to download MiniLM; subsequent runs reuse its cache.
There is no handcrafted synonym map to tune against the labeled questions.

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={"device": "cpu", "revision": MODEL_REVISION},
    encode_kwargs={"normalize_embeddings": True},
)

## Compare Two Chunking Candidates

`whole_documents` keeps each short policy intact, as in notebook 04.
`windowed` uses notebook 01's word windows (`chunk_size=18, overlap=3`).
Both preserve the original `doc_id` for labels and a unique `chunk_id`
for tracing. A source counts as found if any of its chunks is retrieved.

In [ ]:
def window_chunks(chunk_size: int = 18, overlap: int = 3) -> list[dict[str, str]]:
    """Create overlapping candidate chunks while preserving document IDs."""

    if not isinstance(chunk_size, int) or not isinstance(overlap, int):
        raise TypeError("chunk_size and overlap must be integers (word counts).")
    if chunk_size <= 0 or not 0 <= overlap < chunk_size:
        raise ValueError("Use chunk_size > 0 and 0 <= overlap < chunk_size.")
    candidate = []
    # Preserve the document ID so chunk retrieval can be scored against labels.
    step = chunk_size - overlap
    for document in documents:
        words = document["text"].split()
        if not words:
            continue
        for position, start in enumerate(range(0, max(len(words) - overlap, 1), step)):
            candidate.append(
                {
                    **document,
                    "chunk_id": f"{document['doc_id']}-{position}",
                    "text": " ".join(words[start : start + chunk_size]),
                }
            )
    return candidate


windowed_chunks = window_chunks(chunk_size=18, overlap=3)
print(f"Whole-source chunks = {len(chunks)}, window chunks = {len(windowed_chunks)}")

In [ ]:
def build_store(name: str, source_chunks: list[dict[str, str]]) -> QdrantVectorStore:
    source_documents = [
        Document(
            page_content=chunk["text"],
            metadata={key: value for key, value in chunk.items() if key != "text"},
        )
        for chunk in source_chunks
    ]
    return QdrantVectorStore.from_documents(
        source_documents,
        embeddings,
        location=":memory:",
        collection_name=name,
        distance="Cosine",
    )

[Qdrant local mode](https://docs.langchain.com/oss/python/integrations/vectorstores/qdrant)
builds each in-memory collection without a server. This controlled
comparison changes only segmentation, not the embedder or database.
It measures ranking quality, not ANN speed or deployment scalability.

In [ ]:
chunk_sets = {"whole_documents": chunks, "windowed": windowed_chunks}
stores = {name: build_store(name, candidate) for name, candidate in chunk_sets.items()}

In [ ]:
def retrieve_evidence(
    question: str, store: QdrantVectorStore, top_k: int
) -> list[dict[str, Any]]:
    return [
        {
            "doc_id": document.metadata["doc_id"],
            "chunk_id": document.metadata["chunk_id"],
            "score": float(score),
            "excerpt": shorten(document.page_content, width=140),
        }
        for document, score in store.similarity_search_with_score(question, k=top_k)
    ]

## Implement HitRate@k And MRR

Score the first relevant source among the ranked **chunks**. Do not
deduplicate source IDs: several chunks from one policy can consume
several context slots. The metrics support multiple acceptable sources,
though each question here has one label.

We evaluate raw rankings, before notebook 04's safety and similarity
filters, to isolate retrieval failures. These scores do not measure
abstention, complete answer coverage, or generated-answer quality.

In [ ]:
def retrieval_metrics(
    cases: list[dict[str, Any]],
    name: str,
    store: QdrantVectorStore,
    top_k: int = 3,
) -> dict[str, Any]:
    """Measure source ranking and retain per-question evidence."""
    if not cases or top_k < 1:
        raise ValueError("Use a nonempty question set and top_k >= 1.")
    rows: list[dict[str, Any]] = []
    for case in cases:
        evidence = retrieve_evidence(case["question"], store, top_k)
        ranked = [item["doc_id"] for item in evidence]
        ranks = [
            rank
            for rank, doc_id in enumerate(ranked, start=1)
            if doc_id in case["relevant"]
        ]
        rows.append(
            {
                "id": case["id"],
                "question": case["question"],
                "expected": sorted(case["relevant"]),
                "evidence": evidence,
                "ranked": ranked,
                "first_rank": min(ranks) if ranks else None,
            }
        )
    hit_rate = float(np.mean([row["first_rank"] is not None for row in rows]))
    mrr = float(
        np.mean(
            [
                1 / row["first_rank"] if row["first_rank"] is not None else 0
                for row in rows
            ]
        )
    )
    assert 0 <= mrr <= hit_rate <= 1, "Expected 0 <= MRR <= HitRate@k <= 1."
    return {"name": name, "hit_rate_at_k": hit_rate, "mrr": mrr, "rows": rows}

In [ ]:
reports = [
    retrieval_metrics(evaluation_cases, name, store, top_k=3)
    for name, store in stores.items()
]
for report in reports:
    print(
        f"{report['name']}: HitRate@3 = {report['hit_rate_at_k']:.3f}, MRR@3 = {report['mrr']:.3f}"
    )

HitRate@3 asks whether the source reaches a three-chunk context; MRR@3
rewards finding it early. A higher similarity score is not itself an
improvement in either metric, and a source hit does not guarantee that
the retrieved chunk contains every fact needed for the answer.

## Inspect Error Cases

For each question whose expected source is not ranked first, show the expected
source and the actual retrieved chunks, scores, and short excerpts.
Reuse this view for validation errors below.

In [ ]:
def show_failures(report: dict[str, Any]) -> None:
    failures = [row for row in report["rows"] if row["first_rank"] != 1]
    print(f"\n{report['name']}: not-at-rank-1 cases = {len(failures)}")
    for row in failures:
        print(f"{row['id']}: {row['question']}")
        print("Expected:", ", ".join(row["expected"]))
        for rank, item in enumerate(row["evidence"], start=1):
            print(
                f" {rank}. {item['doc_id']} / {item['chunk_id']} | score = {item['score']:.3f}"
            )
            print("   ", item["excerpt"])


for report in reports:
    show_failures(report)

With this small handbook, both candidates find the expected source at
rank one for the development questions. That is useful as a smoke test,
not proof of robustness or evidence that windowing improves retrieval.
Inspect ties and near misses before changing tools. Harder questions,
overlapping policies, and missing answers are needed to expose limits.

## Evaluate Top-k Sensitivity

Context windows have a cost. Increasing `k` can improve hit rate but add
distracting evidence and additional generation tokens. Compare retrieval
quality before deciding how much context the RAG prompt should carry.


In [ ]:
for name, store in stores.items():
    for top_k in [1, 2, 3, 5]:
        report = retrieval_metrics(evaluation_cases, name, store, top_k)
        print(
            f"{name} k={top_k}: HitRate={report['hit_rate_at_k']:.3f}, MRR={report['mrr']:.3f}"
        )

Both candidates already reach every development source at `k=1`.
A larger `k` adds context without improving these labels. Prefer the
smallest tested budget for the next experiment, then check whether
complete answers need more than one passage. Multiple-source questions
would require different labels and an additional coverage metric.

## Check Fresh Validation Questions

Select on development scores at `k=1`, breaking a tie in favor of fewer
indexed chunks. Freeze that choice before checking the four new
questions below. Report validation separately; do not tune the model,
chunking, or `k` to make these questions pass.

In [ ]:
final_reports = [
    retrieval_metrics(evaluation_cases, name, store, top_k=1)
    for name, store in stores.items()
]
best = max(
    final_reports,
    key=lambda report: (
        report["hit_rate_at_k"],
        report["mrr"],
        -len(chunk_sets[report["name"]]),
    ),
)
validation_cases = [
    {
        "id": "v01",
        "question": "Which references are required to reclaim a second charge for one invoice?",
        "relevant": {"refunds"},
    },
    {
        "id": "v02",
        "question": "My authenticator was wiped during a device upgrade. Can support restore access?",
        "relevant": {"mfa-device"},
    },
    {
        "id": "v03",
        "question": "Who authorizes moving a submission date when a project is delayed?",
        "relevant": {"extensions"},
    },
    {
        "id": "v04",
        "question": "Can a customer passport scan be sent to a hosted language model for help?",
        "relevant": {"privacy"},
    },
]
validation_reports = [
    retrieval_metrics(validation_cases, name, store, top_k=1)
    for name, store in stores.items()
]
for report in validation_reports:
    print(
        f"Validation {report['name']}: HitRate@1 = {report['hit_rate_at_k']:.3f}, MRR@1 = {report['mrr']:.3f}"
    )
    show_failures(report)

## From Offline Metrics To A RAG Evaluation Plan

Retrieval metrics answer whether known evidence is found. A production
evaluation also needs additional checks beyond retrieval.

For example, validation question `v04` finds `privacy` at rank one with
whole policies, but its score is below notebook 04's `0.40` cutoff: the
guarded pipeline abstains despite this source hit. Calibrate thresholds on
a separate set, not by retuning them to pass these validation questions.

| Check | Question it answers |
| --- | --- |
| Groundedness | Are answer claims entailed by retrieved passages? |
| Answer relevance | Does the response address the user question? |
| Refusal behavior | Does the pipeline decline unsupported questions? |
| Operational measures | What are the latency, token use, index size, and update cost? |

The Ragas and TruLens ecosystems are examples of tools for
these later layers. They do not remove the need for a carefully labeled
retrieval set.


## Exercise: Make A Shipping Recommendation

Use the development results at `k=1` to write a decision record with:

1. the chunking candidate and embedding model for the next generation experiment,
2. development and separate validation HitRate@1 and MRR@1,
3. one known limitation and one next test.

Keep the frozen development choice even if validation favors another
candidate. Explain why these results are not yet a shipping guarantee.

In [ ]:
# @TODO: Record the frozen candidate, separate metrics, and one next test.

### Reference Solution


In [ ]:
validation = next(
    report for report in validation_reports if report["name"] == best["name"]
)
decision = {
    "selected_retriever": best["name"],
    "embedding_model": MODEL_NAME,
    "embedding_revision": MODEL_REVISION,
    "top_k": 1,
    "development_hit_rate_at_1": round(best["hit_rate_at_k"], 3),
    "development_mrr_at_1": round(best["mrr"], 3),
    "validation_hit_rate_at_1": round(validation["hit_rate_at_k"], 3),
    "validation_mrr_at_1": round(validation["mrr"], 3),
    "known_limitation": "Source-level labels do not prove complete answer coverage; four validation questions are too few for a shipping claim.",
    "next_test": "Add unseen multi-source and unsupported questions, then test the guarded generation path from notebook 04.",
}
print(json.dumps(decision, indent=2))

Equal development scores favor intact policies here because they need
fewer indexed chunks, not because whole documents are universally better.
Keep the validation result, including failures, in the recommendation.
Before production, test notebook 04's evidence filters, abstention, and
generated claims separately; perfect retrieval scores on this small set
cannot establish those properties.

## Extensions

The natural extension is to run notebook 04 generation over these labeled
contexts and evaluate groundedness and answer relevance separately.

## Further Reading

- Pinecone: [Offline Retrieval Evaluation](https://www.pinecone.io/learn/offline-evaluation/)
- Ragas: [Docs](https://docs.ragas.io/en/stable/) / [Metrics](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)
- TruLens: [Docs](https://www.trulens.org/) / [The RAG Triad](https://www.trulens.org/getting_started/core_concepts/rag_triad/)
- [BEIR: A Heterogenous Benchmark for Zero-shot Evaluation of Information Retrieval Models](https://arxiv.org/abs/2104.08663) - Thakur et al., 2021
